# Real-World Exploration: insitro KinDEL (DDR1)

This notebook explores a KinDEL dataset (`data/kindel_ddr1.csv`) using the project’s Bayesian enrichment pipeline.

- **Section 1**: Data integrity (raw count distributions)
- **Section 2**: The “Bayesian Shield” demo (digamma point estimate shrinkage)
- **Section 3**: Scaffold / synthon convergence (enriched families)


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.importer import KinDELImportConfig, import_kindel_counts
from src.analyzer import BetaBinomialConfig, summarize_enrichment, aggregate_enrichment_by_scaffold

sns.set_context("talk")

DATA_PATH = Path("..") / "data" / "kindel_ddr1.csv"
assert DATA_PATH.exists(), f"Missing {DATA_PATH}. Run scripts/setup_real_data.sh first."

raw = pd.read_csv(DATA_PATH)
raw.shape, list(raw.columns)[:20]


In [ ]:
cfg = KinDELImportConfig()
df = import_kindel_counts(raw, config=cfg)

# Keep both the standardized counts and original KinDEL columns for exploration
cols = ["compound_id", cfg.input_col, *cfg.selected_replicate_cols, cfg.output_input_col, cfg.output_selected_col]
cols = [c for c in cols if c in df.columns]

df[cols].head()


## Section 1 — Data Integrity

Plot raw count distributions for input vs. KinDEL selected replicates (before any depth scaling / aggregation).

In [ ]:
input_col = cfg.input_col
rep_cols = list(cfg.selected_replicate_cols)

plot_df = raw[[input_col, *rep_cols]].copy()
for c in [input_col, *rep_cols]:
    plot_df[c] = pd.to_numeric(plot_df[c], errors="coerce")

# Long-form for seaborn
long = plot_df.melt(var_name="pool", value_name="count").dropna()
long["count"] = long["count"].clip(lower=0)

plt.figure(figsize=(10, 5))
sns.histplot(
    data=long,
    x="count",
    hue="pool",
    bins=200,
    element="step",
    stat="density",
    common_norm=False,
    log_scale=(True, True),
)
plt.title("Raw count distributions (log-log)")
plt.xlabel("count")
plt.ylabel("density")
plt.tight_layout()


In [ ]:
# Pool depth summary
pool_totals = {c: float(pd.to_numeric(raw[c], errors="coerce").fillna(0).clip(lower=0).sum()) for c in [input_col, *rep_cols]}
pd.Series(pool_totals).sort_values(ascending=False).to_frame("total_reads")


## Section 2 — The “Bayesian Shield” Demo

Goal: show that the **digamma-based** point estimate shrinks noisy low-count molecules toward 0 enrichment, while preserving high-count signal.

We’ll programmatically find:
- a **low-count** example (input ≈ 1 and selected ≈ 5)
- a **high-count** example (input ≈ 100 and selected ≈ 500)

Then compute Bayesian summaries with `src.analyzer.summarize_enrichment` (digamma point estimate).

In [ ]:
# Find near-target examples by minimizing absolute distance in (input_count, selected_count)

targets = {
    "low": (1, 5),
    "high": (100, 500),
}

work = df[["compound_id", "input_count", "selected_count"]].copy()
work = work[(work["input_count"] >= 0) & (work["selected_count"] >= 0)]

picked = {}
for name, (ti, ts) in targets.items():
    d = (work["input_count"] - ti).abs() + (work["selected_count"] - ts).abs()
    idx = int(d.idxmin())
    picked[name] = work.loc[idx]

picked


In [ ]:
# Compute Bayesian enrichment (digamma point estimate is the primary estimator)

bayes_cfg = BetaBinomialConfig(uncertainty_mode="mc_batched", mc_samples=50_000)
enriched = summarize_enrichment(df, config=bayes_cfg)

examples = enriched[enriched["compound_id"].isin([int(picked["low"]["compound_id"]), int(picked["high"]["compound_id"])])].copy()
examples = examples[[
    "compound_id",
    "input_count",
    "selected_count",
    "log2_enrichment_mean",
    "log2_enrichment_ci_low",
    "log2_enrichment_ci_high",
    "prob_enriched",
]].sort_values("input_count")
examples


In [ ]:
# Visualize the shrinkage: compare raw log2 fold vs Bayesian digamma mean

def raw_log2_fc(x_in: float, x_sel: float, total_in: float, total_sel: float, pc: float = 0.5) -> float:
    return (np.log2((x_sel + pc) / (total_sel + 2 * pc)) - np.log2((x_in + pc) / (total_in + 2 * pc)))

total_in = float(df["input_count"].sum())
total_sel = float(df["selected_count"].sum())

rows = []
for name in ["low", "high"]:
    cid = int(picked[name]["compound_id"])
    r = enriched.loc[enriched["compound_id"] == cid].iloc[0]
    rows.append(
        {
            "example": name,
            "compound_id": cid,
            "raw_log2_fc": raw_log2_fc(r["input_count"], r["selected_count"], total_in, total_sel, pc=0.5),
            "bayes_digamma_mean": float(r["log2_enrichment_mean"]),
        }
    )

comp = pd.DataFrame(rows)
comp


In [ ]:
plt.figure(figsize=(6, 4))
comp_long = comp.melt(id_vars=["example", "compound_id"], value_vars=["raw_log2_fc", "bayes_digamma_mean"], var_name="metric", value_name="log2_enrichment")
sns.barplot(data=comp_long, x="example", y="log2_enrichment", hue="metric")
plt.title("Low-count shrinkage vs high-count preservation")
plt.tight_layout()


## Section 3 — Scaffold Convergence

Goal: find **enriched families** where multiple molecules sharing the same scaffold/synthon show strong Bayesian evidence.

This section looks for a scaffold-like key in the dataset. If the DDR1 table doesn’t include scaffold/synthon columns, you can add one upstream (e.g., scaffold assignment by chemistry pipeline) and re-run this section.

In [ ]:
# Heuristically pick a scaffold/synthon column if present
candidates = [
    "scaffold_id",
    "scaffold",
    "scaffold_hash",
    "synthon",
    "synthon_id",
    "synthon_hash",
    "synthon_1",
    "synthon_2",
    "synthon_3",
]
scaffold_col = next((c for c in candidates if c in enriched.columns), None)
scaffold_col


In [ ]:
if scaffold_col is None:
    print("No scaffold/synthon column detected; skipping family analysis.")
else:
    # Summarize scaffold-level signal with the same Bayesian recipe
    sc = aggregate_enrichment_by_scaffold(enriched, config=bayes_cfg, scaffold_col=scaffold_col)

    # Define an 'enriched family' as a scaffold with multiple members and high scaffold-level evidence
    # (Thresholds are intentionally simple heuristics for exploration.)
    family = (
        sc[(sc["n_members"] >= 3) & (sc["prob_enriched"] >= 0.95)]
        .sort_values(["prob_enriched", "scaffold_log2_enrichment"], ascending=False)
        .head(25)
    )
    display(family.head(10))

    top_scaffolds = set(family[scaffold_col].head(5).tolist())
    members = enriched[enriched[scaffold_col].isin(top_scaffolds)].copy()
    members = members.sort_values([scaffold_col, "log2_enrichment_mean"], ascending=[True, False])

    plt.figure(figsize=(10, 5))
    sns.boxplot(data=members, x=scaffold_col, y="log2_enrichment_mean")
    plt.xticks(rotation=45, ha="right")
    plt.title("Compound enrichment distributions for top enriched families")
    plt.tight_layout()
